# Phase 13 — SFT Evaluation
## CodeMentor-LLM
Evaluating fine-tuned Llama-3.2-3B-Instruct on test split.
Comparing base model vs SFT model using ROUGE and BERTScore.

## Evaluation Steps
1. Load SFT model from HuggingFace Hub
2. Run inference on test split
3. Compute ROUGE-1, ROUGE-2, ROUGE-L
4. Compute BERTScore
5. Compare base vs SFT model
6. Show 10 qualitative examples

# libraries

In [ ]:
!pip uninstall numpy -y
!pip install numpy==1.26.4
!pip install -q transformers==4.49.0 peft==0.14.0 bitsandbytes==0.45.3 accelerate==1.5.1 datasets==3.3.2 bert-score==0.3.13
!pip install -q evaluate==0.4.2 rouge-score==0.1.2 --upgrade

# Login to HuggingFace

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in successfully")

# Load SFT model and tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

model_id = "meta-llama/Llama-3.2-3B-Instruct"
adapter_id = "Abdulmoiz123/codementor-llm-sft"

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config, # load in 4bit
    device_map="auto",
)
print(f"Base model loaded — {base_model.get_memory_footprint() / 1024**3:.2f} GB")

# Load SFT adapter
print("\nLoading SFT adapter...")
sft_model = PeftModel.from_pretrained(base_model, adapter_id)
print("SFT model loaded successfully")

#  Load test dataset

In [ ]:
from datasets import load_dataset

# Load test split
print("Loading test dataset...")
dataset = load_dataset("Abdulmoiz123/codementor-llm-splits")
test_dataset = dataset["test"].select(range(300))

print(f"Test samples: {len(test_dataset)}")
print(f"\nSample:")
print(test_dataset[0]["text"])

# inference function

In [ ]:
SYSTEM_PROMPT = "You are a helpful coding assistant. Answer coding questions clearly and concisely with working code examples."

def generate_response(model, tokenizer, instruction, max_new_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )
    return response

print("Inference function defined")

#  Extract instructions and references from test set

In [ ]:
def extract_instruction(text):
    if "<|start_header_id|>user<|end_header_id|>" in text:
        instruction = text.split("<|start_header_id|>user<|end_header_id|>")[-1]
        instruction = instruction.split("<|eot_id|>")[0].strip()
        return instruction
    return ""

def extract_reference(text):
    if "<|start_header_id|>assistant<|end_header_id|>" in text:
        response = text.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
        response = response.replace("<|eot_id|>", "").strip()
        return response
    return ""

# Extract from test dataset
instructions = [extract_instruction(s["text"]) for s in test_dataset]
references = [extract_reference(s["text"]) for s in test_dataset]

print(f"Instructions extracted: {len(instructions)}")
print(f"References extracted  : {len(references)}")
print(f"\nSample instruction: {instructions[0]}")
print(f"\nSample reference  : {references[0]}")

# Generate SFT model predictions on 50 samples

In [ ]:
sft_predictions = []
for i, instruction in enumerate(instructions[:50]):
    response = generate_response(sft_model, tokenizer, instruction)
    sft_predictions.append(response)
    if (i+1) % 10 == 0:
        print(f"Progress: {i+1}/50")

print(f"\nSFT predictions generated: {len(sft_predictions)}")

# Compute ROUGE scores for SFT model

**ROUGE scores** are evaluation metrics that measure how much a model’s generated text overlaps with a reference text in terms of words and sequences.

Code has many unique tokens so ROUGE scores are naturally lower than text tasks.

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Compute scores for all predictions
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for pred, ref in zip(sft_predictions, references[:50]):
    scores = scorer.score(ref, pred)
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

print("SFT Model ROUGE Scores:")
print(f"  ROUGE-1 : {sum(rouge1_scores)/len(rouge1_scores):.4f}")
print(f"  ROUGE-2 : {sum(rouge2_scores)/len(rouge2_scores):.4f}")
print(f"  ROUGE-L : {sum(rougeL_scores)/len(rougeL_scores):.4f}")

# Generate base model predictions on same 50 samples

In [ ]:
# Disable SFT adapter to get base model predictions
sft_model.disable_adapter_layers()

base_predictions = []
for i, instruction in enumerate(instructions[:50]):
    response = generate_response(sft_model, tokenizer, instruction)
    base_predictions.append(response)
    if (i+1) % 10 == 0:
        print(f"Progress: {i+1}/50")

# Re-enable SFT adapter
sft_model.enable_adapter_layers()

print(f"\nBase predictions generated: {len(base_predictions)}")

#  Compute ROUGE scores for base model

In [ ]:
# Compute base model ROUGE scores
base_rouge1_scores = []
base_rouge2_scores = []
base_rougeL_scores = []

for pred, ref in zip(base_predictions, references[:50]):
    scores = scorer.score(ref, pred)
    base_rouge1_scores.append(scores['rouge1'].fmeasure)
    base_rouge2_scores.append(scores['rouge2'].fmeasure)
    base_rougeL_scores.append(scores['rougeL'].fmeasure)

print("Base Model ROUGE Scores:")
print(f"  ROUGE-1 : {sum(base_rouge1_scores)/len(base_rouge1_scores):.4f}")
print(f"  ROUGE-2 : {sum(base_rouge2_scores)/len(base_rouge2_scores):.4f}")
print(f"  ROUGE-L : {sum(base_rougeL_scores)/len(base_rougeL_scores):.4f}")

print("\nComparison:")
print(f"{'Metric':<12} {'Base':>8} {'SFT':>8} {'Improvement':>12}")
print("-" * 42)
print(f"{'ROUGE-1':<12} {sum(base_rouge1_scores)/len(base_rouge1_scores):>8.4f} {sum(rouge1_scores)/len(rouge1_scores):>8.4f} {(sum(rouge1_scores)/len(rouge1_scores) - sum(base_rouge1_scores)/len(base_rouge1_scores)):>+12.4f}")
print(f"{'ROUGE-2':<12} {sum(base_rouge2_scores)/len(base_rouge2_scores):>8.4f} {sum(rouge2_scores)/len(rouge2_scores):>8.4f} {(sum(rouge2_scores)/len(rouge2_scores) - sum(base_rouge2_scores)/len(base_rouge2_scores)):>+12.4f}")
print(f"{'ROUGE-L':<12} {sum(base_rougeL_scores)/len(base_rougeL_scores):>8.4f} {sum(rougeL_scores)/len(rougeL_scores):>8.4f} {(sum(rougeL_scores)/len(rougeL_scores) - sum(base_rougeL_scores)/len(base_rougeL_scores)):>+12.4f}")

ROUGE-L improvement of +0.05 is significant for a coding dataset trained on only 5K samples.

# Compute BERTScore

BERTScore measures how semantically similar two texts are by comparing their contextual embeddings (meaning), not just exact word matches

In [ ]:
from bert_score import score as bert_score

print("Computing BERTScore for SFT model...")
P_sft, R_sft, F1_sft = bert_score(
    sft_predictions,
    references[:50],
    lang="en",
    verbose=False
)

print("Computing BERTScore for Base model...")
P_base, R_base, F1_base = bert_score(
    base_predictions,
    references[:50],
    lang="en",
    verbose=False
)

print("\nBERTScore Comparison:")
print(f"{'Metric':<12} {'Base':>8} {'SFT':>8} {'Improvement':>12}")
print("-" * 42)
print(f"{'Precision':<12} {P_base.mean():>8.4f} {P_sft.mean():>8.4f} {(P_sft.mean()-P_base.mean()):>+12.4f}")
print(f"{'Recall':<12} {R_base.mean():>8.4f} {R_sft.mean():>8.4f} {(R_sft.mean()-R_base.mean()):>+12.4f}")
print(f"{'F1':<12} {F1_base.mean():>8.4f} {F1_sft.mean():>8.4f} {(F1_sft.mean()-F1_base.mean()):>+12.4f}")

5K samples + 3 epochs is limited — more data would show bigger BERTScore gains

#  Qualitative comparison — 10 examples

In [ ]:
print("QUALITATIVE COMPARISON — 10 EXAMPLES")

for i in range(10):
    print(f"\nExample {i+1}")
    print(f"INSTRUCTION: {instructions[i]}")
    print(f"\nREFERENCE:\n{references[i]}")
    print(f"\nBASE MODEL:\n{base_predictions[i]}")
    print(f"\nSFT MODEL:\n{sft_predictions[i]}")
    print("=" * 60)

# Save evaluation results

In [ ]:
import json

results = {
    "model": "Abdulmoiz123/codementor-llm-sft",
    "base_model": "meta-llama/Llama-3.2-3B-Instruct",
    "test_samples": 50,
    "rouge_scores": {
        "base": {"rouge1": 0.2258, "rouge2": 0.1072, "rougeL": 0.1749},
        "sft":  {"rouge1": 0.2585, "rouge2": 0.1545, "rougeL": 0.2256}
    },
    "bert_scores": {
        "base": {"precision": 0.8037, "recall": 0.8736, "f1": 0.8366},
        "sft":  {"precision": 0.7989, "recall": 0.8798, "f1": 0.8363}
    },
    "observations": {
        "strengths": [
            "Concise direct code answers",
            "Correct syntax for simple tasks",
            "ROUGE scores improved across all metrics"
        ],
        "weaknesses": [
            "Repetitive output loops",
            "Hallucinating outputs on complex tasks",
            "Base model better for explanations"
        ],
        "root_cause": "Only 5K training samples — more data needed"
    }
}

with open("sft_evaluation_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Results saved to sft_evaluation_results.json")